# 03 - KAN NNUE Training

Train KAN-based NNUE variants and compare against the baseline.

**Key question**: Does replacing fixed CReLU activations with trainable B-spline
activations (KAN) improve chess evaluation quality?

**Variants tested**:
1. `KanBoard768` -- Full KAN post-accumulator (both hidden layers replaced)
2. `HybridKanBoard768` -- KAN only on output layer (CReLU retained in hidden)
3. Grid size sweep -- test grid_size={3, 5, 8} to find the sweet spot

---

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q hatchling
!pip install -q git+https://github.com/y0sif/kanue.git
!pip install -q python-chess tqdm

In [ ]:
import torch
import numpy as np
from pathlib import Path

from kanue.models.kan import KanBoard768, HybridKanBoard768
from kanue.data import BulletformatBatchDataset
from kanue.utils import DriveCheckpointer, train_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

## 1. Load Data (same as baseline)

In [ ]:
DATA_DIR = Path('/content/drive/MyDrive/kanue/data')
BATCH_SIZE = 16384

train_loader = BulletformatBatchDataset(
    DATA_DIR / 'train.data', batch_size=BATCH_SIZE, shuffle=True
)
val_loader = BulletformatBatchDataset(
    DATA_DIR / 'val.data', batch_size=BATCH_SIZE, shuffle=False
)

print(f'Train: {len(train_loader.data):,} positions ({len(train_loader)} batches)')
print(f'Val:   {len(val_loader.data):,} positions ({len(val_loader)} batches)')

## 2. Experiment 1: Full KAN Post-Accumulator

In [ ]:
HIDDEN_SIZE = 128
GRID_SIZE = 5
SPLINE_ORDER = 3

kan_model = KanBoard768(
    hidden_size=HIDDEN_SIZE,
    grid_size=GRID_SIZE,
    spline_order=SPLINE_ORDER,
).to(device)

total_params = sum(p.numel() for p in kan_model.parameters())
print(f'KanBoard768(hidden={HIDDEN_SIZE}, grid={GRID_SIZE}, spline_order={SPLINE_ORDER})')
print(f'Total parameters: {total_params:,}')
print()
print(kan_model)

In [ ]:
EPOCHS = 50
LR = 1e-3
LR_DROP_EPOCH = 35

optimizer = torch.optim.Adam(kan_model.parameters(), lr=LR)
checkpointer = DriveCheckpointer(f'kan_g{GRID_SIZE}')

kan_log = train_model(
    model=kan_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
    checkpointer=checkpointer,
    checkpoint_every=5,
    lr_drop_epoch=LR_DROP_EPOCH,
)

print(f'\nBest val loss: {min(kan_log["val_loss"]):.6f}')
print(f'Best val accuracy: {max(kan_log["val_accuracy"]):.4f}')

## 3. Experiment 2: Hybrid KAN (CReLU + KAN output)

In [ ]:
hybrid_model = HybridKanBoard768(
    hidden_size=HIDDEN_SIZE,
    grid_size=GRID_SIZE,
    spline_order=SPLINE_ORDER,
).to(device)

total_params = sum(p.numel() for p in hybrid_model.parameters())
print(f'HybridKanBoard768(hidden={HIDDEN_SIZE}, grid={GRID_SIZE})')
print(f'Total parameters: {total_params:,}')

optimizer = torch.optim.Adam(hybrid_model.parameters(), lr=LR)
checkpointer = DriveCheckpointer(f'hybrid_kan_g{GRID_SIZE}')

hybrid_log = train_model(
    model=hybrid_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
    checkpointer=checkpointer,
    checkpoint_every=5,
    lr_drop_epoch=LR_DROP_EPOCH,
)

print(f'\nBest val loss: {min(hybrid_log["val_loss"]):.6f}')
print(f'Best val accuracy: {max(hybrid_log["val_accuracy"]):.4f}')

## 4. Experiment 3: Grid Size Sweep

In [ ]:
grid_sweep_logs = {}

for grid_size in [3, 5, 8]:
    print(f'\n{"="*60}')
    print(f'Training KanBoard768 with grid_size={grid_size}')
    print(f'{"="*60}')

    model = KanBoard768(
        hidden_size=HIDDEN_SIZE,
        grid_size=grid_size,
        spline_order=SPLINE_ORDER,
    ).to(device)

    params = sum(p.numel() for p in model.parameters())
    print(f'Parameters: {params:,}')

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    checkpointer = DriveCheckpointer(f'kan_sweep_g{grid_size}')

    log = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        device=device,
        epochs=EPOCHS,
        checkpointer=checkpointer,
        checkpoint_every=10,
        lr_drop_epoch=LR_DROP_EPOCH,
    )

    grid_sweep_logs[grid_size] = log
    print(f'grid_size={grid_size}: best_val_loss={min(log["val_loss"]):.6f}, best_acc={max(log["val_accuracy"]):.4f}')

## 5. Compare All Variants

In [ ]:
import matplotlib.pyplot as plt
import json

# Load baseline log from Drive
baseline_checkpointer = DriveCheckpointer('baseline')
baseline_log = baseline_checkpointer.load_log()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Val loss comparison
if baseline_log:
    axes[0].plot(baseline_log['val_loss'], label='Baseline NNUE', linewidth=2)
axes[0].plot(kan_log['val_loss'], label=f'Full KAN (grid={GRID_SIZE})', linewidth=2)
axes[0].plot(hybrid_log['val_loss'], label=f'Hybrid KAN (grid={GRID_SIZE})', linewidth=2)
for gs, log in grid_sweep_logs.items():
    if gs != GRID_SIZE:  # Avoid duplicate
        axes[0].plot(log['val_loss'], label=f'KAN grid={gs}', alpha=0.7, linestyle='--')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Val MSE Loss')
axes[0].set_title('Validation Loss')
axes[0].legend()

# Val accuracy comparison
if baseline_log:
    axes[1].plot(baseline_log['val_accuracy'], label='Baseline NNUE', linewidth=2)
axes[1].plot(kan_log['val_accuracy'], label=f'Full KAN (grid={GRID_SIZE})', linewidth=2)
axes[1].plot(hybrid_log['val_accuracy'], label=f'Hybrid KAN (grid={GRID_SIZE})', linewidth=2)
for gs, log in grid_sweep_logs.items():
    if gs != GRID_SIZE:
        axes[1].plot(log['val_accuracy'], label=f'KAN grid={gs}', alpha=0.7, linestyle='--')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].legend()

plt.suptitle('KAN vs Baseline NNUE', fontweight='bold')
plt.tight_layout()

results_dir = Path('/content/drive/MyDrive/kanue/results')
plt.savefig(str(results_dir / 'kan_vs_baseline.png'), dpi=150)
plt.show()

In [ ]:
# Summary table
print(f'{"Model":<30} {"Best Val Loss":>15} {"Best Val Acc":>15} {"Params":>12}')
print('-' * 75)

if baseline_log:
    print(f'{"Baseline NNUE":<30} {min(baseline_log["val_loss"]):>15.6f} {max(baseline_log["val_accuracy"]):>15.4f} {"~99k":>12}')

print(f'{f"Full KAN (grid={GRID_SIZE})":<30} {min(kan_log["val_loss"]):>15.6f} {max(kan_log["val_accuracy"]):>15.4f}')
print(f'{f"Hybrid KAN (grid={GRID_SIZE})":<30} {min(hybrid_log["val_loss"]):>15.6f} {max(hybrid_log["val_accuracy"]):>15.4f}')

for gs, log in sorted(grid_sweep_logs.items()):
    print(f'{f"KAN sweep grid={gs}":<30} {min(log["val_loss"]):>15.6f} {max(log["val_accuracy"]):>15.4f}')

print('\nKAN training complete. Proceed to notebook 04 for detailed analysis.')